In [90]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import HTML, display 
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import accuracy_score,r2_score,precision_score,recall_score,f1_score,roc_auc_score
import pickle

PALETTE   = "YlGn"
ACCENT    = "#10b981"
DARK      = "#0f2d24"

sns.set_theme(style="whitegrid", palette="YlGn")
plt.rcParams.update({
    "figure.figsize":    (13, 5),
    "axes.titlesize":    14,
    "axes.titleweight":  "bold",
    "axes.titlecolor":   DARK,
    "axes.labelsize":    11,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.facecolor":  "white",
    "axes.facecolor":    "#fafafa",
    "grid.color":        "#e5e7eb",
})
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [91]:
train_df=pd.read_csv('D:\\DS Part(2)\\F1_pitstop\\playground-series-s6e5\\train.csv')
test_df=pd.read_csv('D:\\DS Part(2)\\F1_pitstop\\playground-series-s6e5\\test.csv')
test_id=test_df['id']
train_df=train_df.drop('id',axis=1)
test_df=test_df.drop('id',axis=1)

In [92]:
train_df
test_df 

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
0,D119,MEDIUM,British Grand Prix,2023,0,21,1,21.0000,4,93.3870,0.2800,-4.9840,0.4038,0.0000
1,VER,MEDIUM,Abu Dhabi Grand Prix,2023,0,24,1,24.0000,1,90.8670,-0.1290,-1.9900,0.4138,0.0000
2,D270,MEDIUM,British Grand Prix,2023,0,24,1,24.0000,11,92.8710,0.0410,-8.8420,0.4615,0.0000
3,D112,SOFT,São Paulo Grand Prix,2024,0,6,2,4.0000,15,94.9670,-19.7410,8.2500,0.0779,1.0000
4,AND,HARD,United States Grand Prix,2024,0,52,2,29.0000,12,99.1120,0.9300,-20.8480,0.7222,7.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188160,D171,MEDIUM,Australian Grand Prix,2024,1,14,1,14.0000,4,83.8790,-16.9190,-87.7670,0.1795,-2.0000
188161,RUS,SOFT,Pre-Season Testing,2025,0,60,3,26.0000,4,95.7270,7.9200,-36.4850,0.7895,-3.0000
188162,D112,MEDIUM,Hungarian Grand Prix,2022,0,28,2,21.0000,7,85.0580,-14.1800,-0.3390,0.3889,3.0000
188163,D349,MEDIUM,Monaco Grand Prix,2024,0,20,2,15.0000,7,80.0740,-19.0040,-37.9670,0.2564,0.0000


In [93]:
train_df['WearRate'] = (
    train_df['Cumulative_Degradation'] /
    (train_df['TyreLife'] + 1)
)

test_df['WearRate'] = (
    test_df['Cumulative_Degradation'] /
    (test_df['TyreLife'] + 1)
)

train_df['PaceDrop'] = (
    train_df['LapTime_Delta'] /
    (train_df['TyreLife'] + 1)
)

test_df['PaceDrop'] = (
    test_df['LapTime_Delta'] /
    (test_df['TyreLife'] + 1)
)

train_df['TyreStress'] = (
    train_df['TyreLife'] *
    train_df['LapTime_Delta']
)

test_df['TyreStress'] = (
    test_df['TyreLife'] *
    test_df['LapTime_Delta']
)

train_df['PositionPressure'] = (
    train_df['Position'] *
    train_df['LapTime_Delta']
)

test_df['PositionPressure'] = (
    test_df['Position'] *
    test_df['LapTime_Delta']
)

train_df['RaceFatigue'] = (
    train_df['RaceProgress'] *
    train_df['TyreLife']
)

test_df['RaceFatigue'] = (
    test_df['RaceProgress'] *
    test_df['TyreLife']
)

train_df.drop('LapNumber', axis=1, inplace=True)
test_df.drop('LapNumber', axis=1, inplace=True)


In [94]:
train_df.drop('PitStop', axis=1, inplace=True)
test_df.drop('PitStop', axis=1, inplace=True)

In [95]:
X = train_df.drop('PitNextLap', axis=1)
y = train_df['PitNextLap']

In [96]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [97]:
num_features=X.select_dtypes(exclude='object').columns
cat_features=X.select_dtypes(include='object').columns


In [98]:


num_pipeline=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])

cat_pipeline=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy="most_frequent")),
    ('Encoding',OneHotEncoder(handle_unknown='ignore'))
])

preprocessor=ColumnTransformer(
    transformers=[
        ('num',num_pipeline,num_features),
        ('cat',cat_pipeline,cat_features)
    ]
)




In [99]:
from catboost import CatBoostClassifier


In [100]:
models={
    # 'Decision Tree':DecisionTreeClassifier(),
    'RF':RandomForestClassifier(n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1),
    'XGBOOST':XGBClassifier(n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'),
    # 'ADABOOST':AdaBoostClassifier(),
    # 'Cat':CatBoostClassifier(verbose=0)
}

In [101]:
model_list = []
accuracy_list = []
roc_list = []

best_accuracy = 0
best_pipe = None

for name, model in models.items():

    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # Train
    pipe.fit(X_train, y_train)

    # Predictions
    y_train_pred = pipe.predict(X_train)
    y_test_pred = pipe.predict(X_test)

    # Probability predictions
    y_prob = pipe.predict_proba(X_test)[:, 1]

    # Metrics
    acc = accuracy_score(y_test, y_test_pred)

    roc = roc_auc_score(y_test, y_prob)

    f1 = f1_score(y_test, y_test_pred, average='weighted')

    precision = precision_score(
        y_test,
        y_test_pred,
        average='weighted'
    )

    recall = recall_score(
        y_test,
        y_test_pred,
        average='weighted'
    )

    # Store results
    model_list.append(name)

    accuracy_list.append(acc)

    roc_list.append(roc)

    # Best model selection
    if acc > best_accuracy:
        
        best_accuracy = acc
        
        best_pipe = pipe

    # Print Results
    print(f"\n{name}")

    print("Training Data:")
    print("- Accuracy:", accuracy_score(y_train, y_train_pred))
    print("- F1:", f1_score(y_train, y_train_pred, average='weighted'))
    print("- Precision:", precision_score(y_train, y_train_pred, average='weighted'))
    print("- Recall:", recall_score(y_train, y_train_pred, average='weighted'))

    print("\nTesting Data:")
    print("- Accuracy:", acc)
    print("- F1:", f1)
    print("- Precision:", precision)
    print("- Recall:", recall)
    print("- ROC AUC:", roc)

    print("=" * 50)


RF
Training Data:
- Accuracy: 0.8196019492644715
- F1: 0.7608020903664217
- Precision: 0.8203539640093439
- Recall: 0.8196019492644715

Testing Data:
- Accuracy: 0.8196019492644715
- F1: 0.7608040560622361
- Precision: 0.8203396114403025
- Recall: 0.8196019492644715
- ROC AUC: 0.8952259775425858

XGBOOST
Training Data:
- Accuracy: 0.9018336976818326
- F1: 0.9013787080436971
- Precision: 0.9009822580218252
- Recall: 0.9018336976818326

Testing Data:
- Accuracy: 0.8947829849250808
- F1: 0.8941408362827419
- Precision: 0.8935975914096842
- Recall: 0.8947829849250808
- ROC AUC: 0.9456385490243535


In [102]:
from sklearn.model_selection import StratifiedKFold,cross_val_score

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    best_pipe,
    X,
    y,
    cv=skf,
    scoring='accuracy'
)

In [103]:
results_df = pd.DataFrame({

    'Model': model_list,

    'Accuracy': accuracy_list,

    'ROC_AUC': roc_list

})

print(
    results_df.sort_values(
        by='Accuracy',
        ascending=False
    )
)

     Model  Accuracy  ROC_AUC
1  XGBOOST    0.8948   0.9456
0       RF    0.8196   0.8952


In [104]:
best_model_name = results_df.sort_values(by='Accuracy', ascending=False).iloc[0]['Model']
print(f"Best Model: {best_model_name}")
print(f"Best Accuracy: {best_accuracy:.4f}")

Best Model: XGBOOST
Best Accuracy: 0.8948


In [112]:
best_pipe.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [113]:
with open('best_model.pkl', 'wb') as f:
    pickle.dump(best_pipe, f)

In [114]:
test_prob = best_pipe.predict_proba(test_df)[:,1]

# Threshold tuning
test_predictions = (
    test_prob > 0.5
).astype(int)

In [115]:
submission = pd.DataFrame({
    'id': test_id,
    'PitNextLap': test_predictions
})


In [116]:
submission.to_csv('submission3.csv', index=False)